# 에너지캐시백 임베딩 모델 평가 + Tool-use Agent 프로토타입

전력소모패턴 · 가전별 상태 · 날씨/가구 메타데이터를 텍스트로 변환해
3개 임베딩 모델로 인코딩하고, Precision@5로 최적 모델을 선정합니다.
cell-8에서는 선정된 임베딩을 유사 가구 검색 툴로 연결한
OpenAI Function Calling 기반 Agent 프로토타입을 실행합니다.

**임베딩 모델 3종**
| 약칭 | 모델 ID |
|------|---------|
| E5-small | `intfloat/multilingual-e5-small` |
| Jina-v5-nano | `jinaai/jina-embeddings-v5-text-nano` |
| Harrier-270m | `microsoft/harrier-oss-v1-270m` |

**실행 순서**
1. cell-1: 패키지 설치
2. cell-2: Google Drive 마운트
3. cell-3~5: 설정·데이터 로드·텍스트 변환
4. cell-6: 3개 모델 임베딩 구축
5. cell-7: Precision@5 벤치마크 → 최적 모델 선정
6. cell-8: Tool-use Agent 프로토타입 (Function Calling + 유사 가구 검색)

**실행 전 확인사항**
1. 런타임 → GPU 선택
2. Google Drive에 학습데이터-라벨링데이터 폴더 업로드
3. Colab Secrets에 `OPENAI_API_KEY` 등록

In [ ]:
# ── 패키지 설치 ───────────────────────────────────────────────
!pip install -q sentence-transformers openai

In [20]:
# ── Google Drive 마운트 ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# 실제 경로로 수정
DATA_DIR = '/content/drive/MyDrive/학습데이터-라벨링데이터'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── 임포트 + 설정 ─────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from datetime import date, datetime
from pathlib import Path
from google.colab import userdata

# ── 설정값 ────────────────────────────────────────────────────
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

TARGET_HOUSES = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d)) and d.startswith("house_")
])
SKIP_APPLIANCES = {"메인 분전반", "unknown"}
TOP_K           = 5

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"대상 가구: {len(TARGET_HOUSES)}가구")
print(f"예시: {TARGET_HOUSES[:5]}")

In [ ]:
# ── 데이터 로더 (1시간 단위 전력 프로파일 + 가전 시간별 에너지 + 날씨/가구 메타데이터) ──
@dataclass
class HouseDayData:
    house_id: str
    day: date
    profile_24: np.ndarray          # 24시간 단위 에너지 (Wh/시간)
    temperature: float | None
    windchill: float | None
    humidity: float | None
    weather: str | None
    is_weekday: bool
    house_type: str | None
    residential_type: str | None
    residential_area: float | None
    members: int | None
    appliance_text: str = ''        # "에어컨 14시 5.20kWh, 밥솥 07시 0.80kWh, ..."


def load_house_data(house_id: str) -> list[HouseDayData]:
    house_path = Path(DATA_DIR) / house_id
    if not house_path.exists():
        print(f'  [경고] {house_id} 없음')
        return []

    temp_map, wc_map, hum_map, weather_map = {}, {}, {}, {}
    house_type = residential_type = None
    residential_area = members = None

    ch01 = house_path / 'ch01.parquet'
    if ch01.exists():
        df = pd.read_parquet(ch01)
        if 'house_type' in df.columns:
            v = df['house_type'].dropna()
            house_type = str(v.iloc[0]) if len(v) else None
        if 'residential_type' in df.columns:
            v = df['residential_type'].dropna()
            residential_type = str(v.iloc[0]) if len(v) else None
        if 'residential_area' in df.columns:
            v = pd.to_numeric(df['residential_area'], errors='coerce').dropna()
            residential_area = float(v.iloc[0]) if len(v) else None
        if 'members' in df.columns:
            v = pd.to_numeric(df['members'], errors='coerce').dropna()
            members = int(v.iloc[0]) if len(v) else None
        for _, row in df.iterrows():
            key = str(pd.to_datetime(row['date']).date())
            t   = pd.to_numeric(row.get('temperature', None), errors='coerce')
            if not pd.isna(t): temp_map[key] = float(t)
            wc  = pd.to_numeric(row.get('windchill', None), errors='coerce')
            if not pd.isna(wc): wc_map[key] = float(wc)
            hum = pd.to_numeric(row.get('humidity', None), errors='coerce')
            if not pd.isna(hum): hum_map[key] = float(hum)
            if 'weather' in row.index and pd.notna(row['weather']):
                weather_map[key] = str(row['weather'])

    # 1시간 단위 전력 프로파일 + 가전별 시간 에너지
    day_profiles:   dict[date, np.ndarray]            = {}  # {date: 24-Wh}
    day_app_hourly: dict[date, dict[str, np.ndarray]] = {}  # {date: {name: 24-Wh}}

    for ch in range(2, 24):
        p = house_path / f'ch{ch:02d}.parquet'
        if not p.exists():
            continue
        df = pd.read_parquet(p)
        name = str(df['name'].iloc[0]) if 'name' in df.columns else f'ch{ch:02d}'
        if name in SKIP_APPLIANCES:
            continue
        df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
        df['end_time']   = pd.to_datetime(df['end_time'],   errors='coerce')
        df['power_w']    = pd.to_numeric(df['power_consumption'], errors='coerce').fillna(0)
        df = df.dropna(subset=['start_time'])

        for _, row in df.iterrows():
            d  = row['start_time'].date()
            pw = float(row['power_w'])
            if pw < 1:
                continue
            st_min = int(row['start_time'].hour * 60 + row['start_time'].minute)
            et_ts  = pd.to_datetime(row['end_time'], errors='coerce')
            et_min = int(et_ts.hour * 60 + et_ts.minute) if pd.notna(et_ts) else st_min + 1
            st_min = max(0, min(st_min, 1439))
            et_min = max(st_min + 1, min(et_min, 1440))

            if d not in day_profiles:
                day_profiles[d]   = np.zeros(24, dtype=np.float32)
                day_app_hourly[d] = {}
            if name not in day_app_hourly[d]:
                day_app_hourly[d][name] = np.zeros(24, dtype=np.float32)

            # 이벤트 에너지를 해당 시간 버킷에 분배 (Wh)
            for h_idx in range(st_min // 60, min(et_min // 60 + 1, 24)):
                h_start = h_idx * 60
                overlap = max(0, min(et_min, h_start + 60) - max(st_min, h_start))
                if overlap > 0:
                    wh = pw * overlap / 60
                    day_profiles[d][h_idx]         += wh
                    day_app_hourly[d][name][h_idx] += wh

    result = []
    for day, prof in sorted(day_profiles.items()):
        dkey  = str(day)
        is_wd = datetime.combine(day, datetime.min.time()).weekday() < 5
        # 가전별 일간 kWh + 피크시간 → "에어컨 14시 5.20kWh, 밥솥 07시 0.80kWh, ..."
        app_data    = day_app_hourly.get(day, {})
        sorted_apps = sorted(
            [(n, arr) for n, arr in app_data.items() if arr.sum() >= 10],
            key=lambda x: x[1].sum(), reverse=True
        )[:8]
        app_parts = [
            f'{n} {int(np.argmax(arr))}시 {arr.sum()/1000:.2f}kWh'
            for n, arr in sorted_apps
        ]
        app_text = ', '.join(app_parts) if app_parts else '사용 없음'

        result.append(HouseDayData(
            house_id=house_id, day=day, profile_24=prof,
            temperature=temp_map.get(dkey),
            windchill=wc_map.get(dkey),
            humidity=hum_map.get(dkey),
            weather=weather_map.get(dkey),
            is_weekday=is_wd,
            house_type=house_type,
            residential_type=residential_type,
            residential_area=residential_area,
            members=members,
            appliance_text=app_text,
        ))
    return result


print('데이터 로딩 중...')
all_data: dict[str, list[HouseDayData]] = {}
for h in TARGET_HOUSES:
    d = load_house_data(h)
    if d:
        all_data[h] = d
        meta = d[0]
        print(f'  {h}: {len(d)}일 | {meta.house_type or "주택?"} {meta.members or "?"}명 | 예시: {meta.appliance_text[:60]}')
print(f'완료: {len(all_data)}가구')


In [ ]:
# ── 텍스트 변환 유틸리티 + 데이터 사전 구축 ───────────────────────────────
import numpy as np
from datetime import datetime

WEEKDAY_KO = ['월', '화', '수', '목', '금', '토', '일']
SEASON_KO  = {3:'봄',4:'봄',5:'봄',6:'여름',7:'여름',8:'여름',
              9:'가을',10:'가을',11:'가을'}


def get_season(d) -> int:
    return {3:0,4:0,5:0,6:1,7:1,8:1,9:2,10:2,11:2}.get(d.month, 3)

def get_season_name(d) -> str:
    return ['봄','여름','가을','겨울'][get_season(d)]

def appliance_names(text: str) -> set:
    """'가전명 HH시 X.XXkWh, ...' 형식에서 가전명 집합 추출."""
    names = set()
    for token in text.split(','):
        parts, name_parts = token.strip().split(), []
        for p in parts:
            if p.endswith('시') and p[:-1].isdigit(): break
            if p.endswith('kWh'): break
            name_parts.append(p)
        if name_parts:
            names.add(' '.join(name_parts))
    return names


def profile_to_text(profile_24: np.ndarray,
                    temperature: float | None = None,
                    humidity:    float | None = None) -> str:
    """시간별 Wh 배열 → 전력 패턴 설명 텍스트."""
    total_kwh = float(profile_24.sum()) / 1000
    peak_h    = int(np.argmax(profile_24))

    active = sorted(
        [(h, float(profile_24[h])) for h in range(24) if profile_24[h] >= 10],
        key=lambda x: x[1], reverse=True
    )[:3]
    top_str = ', '.join(f'{h}시 {w/1000:.2f}kWh' for h, w in active) or '사용 없음'

    blocks = [
        ('새벽(0-6시)',   profile_24[0:6].sum()),
        ('오전(6-12시)',  profile_24[6:12].sum()),
        ('오후(12-18시)', profile_24[12:18].sum()),
        ('저녁(18-22시)', profile_24[18:22].sum()),
        ('심야(22-24시)', profile_24[22:24].sum()),
    ]
    dominant = max(blocks, key=lambda x: x[1])[0] if total_kwh > 0 else '없음'

    parts = [
        f'일사용량 {total_kwh:.2f}kWh',
        f'피크 {peak_h}시',
        f'주사용 {dominant}',
        f'상위시간: {top_str}',
    ]
    if temperature is not None: parts.append(f'기온 {temperature:.1f}℃')
    if humidity    is not None: parts.append(f'습도 {humidity:.0f}%')
    return ' | '.join(parts)


def meta_to_text(d) -> str:
    """HouseDayData → 메타데이터 설명 텍스트."""
    season = SEASON_KO.get(d.day.month, '겨울')
    wd     = '평일' if d.is_weekday else '주말'
    parts  = [f'{season} {wd}']
    if d.house_type:       parts.append(d.house_type)
    if d.residential_type: parts.append(d.residential_type)
    if d.residential_area: parts.append(f'{d.residential_area:.0f}㎡')
    if d.members:          parts.append(f'{d.members}인 거주')
    if d.weather:          parts.append(d.weather)
    return ' '.join(parts)


# ── 전체 텍스트 + 조회 사전 구축 ──────────────────────────────────────────
all_keys_ordered: list[tuple] = []
all_texts:        dict[tuple, dict[str, str]] = {}
_day_lookup:      dict[tuple, object]          = {}

for hid, hd in all_data.items():
    for d in hd:
        key = (hid, d.day)
        all_keys_ordered.append(key)
        all_texts[key] = {
            'power':     profile_to_text(d.profile_24, d.temperature, d.humidity),
            'appliance': d.appliance_text or '사용 없음',
            'meta':      meta_to_text(d),
        }
        _day_lookup[key] = d

print(f'변환 완료: {len(all_texts)}개 (house × day)')
ex = all_texts[all_keys_ordered[0]]
print(f'\n예시:')
print(f'  [전력] {ex["power"]}')
print(f'  [가전] {ex["appliance"][:80]}')
print(f'  [메타] {ex["meta"]}')


In [ ]:
# ── 3개 임베딩 모델 × 3개 타입(전력·가전·메타) → 가중 결합 임베딩 ──────────
import subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sentence_transformers import SentenceTransformer

subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], capture_output=True)
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 설정 ─────────────────────────────────────────────────────────────────
MODEL_LIST = [
    ('intfloat/multilingual-e5-small',      'E5-small',     {}),
    ('jinaai/jina-embeddings-v5-text-nano', 'Jina-v5-nano', {'model_kwargs': {'default_task': 'retrieval'}}),
    ('microsoft/harrier-oss-v1-270m',       'Harrier-270m', {}),
]

# 전력 : 가전 : 메타 결합 가중치
W_POWER, W_APPLIANCE, W_META = 0.4, 0.4, 0.2


def encode_texts(model, texts: list[str], model_id: str) -> np.ndarray:
    """E5 계열은 'query: ' 접두사 적용 후 배치 임베딩."""
    prefix = 'query: ' if 'e5' in model_id.lower() else ''
    return model.encode(
        [f'{prefix}{t}' for t in texts],
        normalize_embeddings=True, batch_size=32, show_progress_bar=False,
    )


def build_combined(pw_embs, app_embs, mt_embs,
                   w_pw=W_POWER, w_app=W_APPLIANCE, w_mt=W_META) -> np.ndarray:
    """단위벡터 3개 가중 합산 후 재정규화."""
    combined = w_pw * pw_embs + w_app * app_embs + w_mt * mt_embs
    norms    = np.linalg.norm(combined, axis=1, keepdims=True) + 1e-9
    return combined / norms


# ── 모델별 임베딩 구축 ────────────────────────────────────────────────────
N = len(all_keys_ordered)
power_texts     = [all_texts[k]['power']     for k in all_keys_ordered]
appliance_texts = [all_texts[k]['appliance'] for k in all_keys_ordered]
meta_texts      = [all_texts[k]['meta']      for k in all_keys_ordered]

loaded_models:    dict[str, SentenceTransformer] = {}
combined_embs:    dict[str, dict[tuple, np.ndarray]] = {}  # model → {key: emb}
component_embs:   dict[str, dict[str, dict[tuple, np.ndarray]]] = {}  # model → comp → {key: emb}

for model_id, name, init_kw in MODEL_LIST:
    print(f'\n[{name}] 로딩 중...')
    try:
        model = SentenceTransformer(model_id, trust_remote_code=True, **init_kw)
        loaded_models[name] = model

        pw_embs  = encode_texts(model, power_texts,     model_id)  # (N, dim)
        app_embs = encode_texts(model, appliance_texts, model_id)
        mt_embs  = encode_texts(model, meta_texts,      model_id)

        comb = build_combined(pw_embs, app_embs, mt_embs)

        combined_embs[name]  = {k: comb[i]    for i, k in enumerate(all_keys_ordered)}
        component_embs[name] = {
            'power':     {k: pw_embs[i]  / (np.linalg.norm(pw_embs[i])  + 1e-9) for i, k in enumerate(all_keys_ordered)},
            'appliance': {k: app_embs[i] / (np.linalg.norm(app_embs[i]) + 1e-9) for i, k in enumerate(all_keys_ordered)},
            'meta':      {k: mt_embs[i]  / (np.linalg.norm(mt_embs[i])  + 1e-9) for i, k in enumerate(all_keys_ordered)},
        }
        print(f'  완료: {N}개, 차원={comb.shape[1]}, 가중치=({W_POWER}/{W_APPLIANCE}/{W_META})')

    except Exception as e:
        print(f'  오류: {e}')

print(f'\n임베딩 완료: {list(combined_embs.keys())}')


In [ ]:
# ── 벤치마크 — 모델 × 컴포넌트 Precision@5 비교표 ──────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

# ── 평가 함수 ──────────────────────────────────────────────────────────────
def cosine_retrieve(query_key, emb_dict, top_k=TOP_K,
                    season_filter=True, weekday_filter=True) -> list:
    """단일 임베딩 기반 코사인 유사도 검색 (시즌·요일 필터)."""
    q_emb    = emb_dict.get(query_key)
    q_day    = query_key[1]
    q_season = get_season(q_day)
    q_is_wd  = datetime.combine(q_day, datetime.min.time()).weekday() < 5

    candidates = [
        k for k in emb_dict
        if k != query_key
        and (not season_filter  or get_season(k[1]) == q_season)
        and (not weekday_filter or
             (datetime.combine(k[1], datetime.min.time()).weekday() < 5) == q_is_wd)
    ]
    if len(candidates) < top_k:
        candidates = [k for k in emb_dict if k != query_key]

    if q_emb is None:
        return candidates[:top_k]

    scores = {k: float(q_emb @ emb_dict[k]) for k in candidates}
    return sorted(scores, key=scores.get, reverse=True)[:top_k]


def precision_at_k(emb_dict, top_k=TOP_K, tol=0.3) -> float:
    """Precision@K — 검색된 타가구 일사용량이 쿼리 ±tol 이내인 비율."""
    hits_list = []
    for qk, q_d in _day_lookup.items():
        if qk not in emb_dict:
            continue
        q_kwh = float(q_d.profile_24.sum()) / 1000
        if q_kwh == 0:
            continue
        retrieved = cosine_retrieve(qk, emb_dict, top_k)
        hits = sum(
            1 for r_hid, r_day in retrieved
            if r_hid != qk[0]
            and (r_d := _day_lookup.get((r_hid, r_day))) is not None
            and abs(float(r_d.profile_24.sum()) / 1000 - q_kwh) / (q_kwh + 1e-9) <= tol
        )
        hits_list.append(hits / top_k)
    return round(float(np.mean(hits_list)), 4) if hits_list else 0.0


# ── 전체 벤치마크 실행 ───────────────────────────────────────────────────
COMP_LABELS = {'power': '전력', 'appliance': '가전', 'meta': '메타'}
rows = []

print('벤치마크 실행 중...')
for model_name in combined_embs:
    # 컴포넌트 단독
    for comp, label in COMP_LABELS.items():
        emb = component_embs[model_name][comp]
        p5  = precision_at_k(emb)
        rows.append({'모델': model_name, '임베딩': label, 'Precision@5': p5})
        print(f'  {model_name:15s} {label:4s}: {p5:.4f}')

    # 결합 (전력+가전+메타)
    p5 = precision_at_k(combined_embs[model_name])
    rows.append({'모델': model_name, '임베딩': f'결합({W_POWER}/{W_APPLIANCE}/{W_META})', 'Precision@5': p5})
    print(f'  {model_name:15s} 결합  : {p5:.4f}')

# ── 결과 테이블 ──────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
pivot = df.pivot(index='임베딩', columns='모델', values='Precision@5')
print('\n=== Precision@5 비교표 (↑ 높을수록 유사사례 검색 품질 우수) ===')
print(pivot.to_string())

# ── 시각화 ───────────────────────────────────────────────────────────────
ax = pivot.plot(kind='bar', figsize=(10, 5), width=0.6)
ax.set_title('임베딩 모델 × 컴포넌트별 Precision@5', fontsize=12)
ax.set_ylabel('Precision@5')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=15)
ax.legend(title='모델', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('embedding_benchmark.png', dpi=150)
plt.show()

# ── 최적 모델 선택 → 이후 셀에 적용 ────────────────────────────────────
combined_rows = df[df['임베딩'].str.startswith('결합')]
best_model_name   = combined_rows.loc[combined_rows['Precision@5'].idxmax(), '모델']
best_combined_emb = combined_embs[best_model_name]
all_keys          = list(best_combined_emb.keys())

print(f'\n★ 최적 모델: {best_model_name}')
print(f'  결합 Precision@5 = {combined_rows.loc[combined_rows["모델"]==best_model_name, "Precision@5"].values[0]:.4f}')
print(f'  → cell-8 이후 RAG 파이프라인에 자동 적용')


In [ ]:
# ── Tool-use Agent 프로토타입 ── OpenAI Function Calling + 임베딩 유사 가구 검색 ──
import json, textwrap
from datetime import date
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

# ═══════════════════════════════════════════════════════════════════════════
# 1. Mock 툴 구현 — cell-4/5/7 에서 로드된 데이터 구조 재사용
# ═══════════════════════════════════════════════════════════════════════════

def _resolve_key(household_id: str, date_str: str | None = None):
    """household_id + 선택적 날짜 → (house_id, date) 키. date_str 없으면 최신 일자."""
    days = all_data.get(household_id, [])
    if not days:
        return None
    if date_str:
        try:
            target = date.fromisoformat(date_str)
            for d in days:
                if d.day == target:
                    return (household_id, d.day)
        except ValueError:
            pass
    return (household_id, days[-1].day)


def get_household_profile(household_id: str) -> dict:
    """가구 메타 + 최근 30일 평균 소비."""
    days = all_data.get(household_id)
    if not days:
        return {"summary": f"{household_id} 데이터 없음", "raw": {}}
    meta = days[0]
    avg_kwh = float(np.mean([d.profile_24.sum() / 1000 for d in days[-30:]]))
    raw = {
        "household_id": household_id,
        "house_type": meta.house_type,
        "residential_type": meta.residential_type,
        "residential_area": meta.residential_area,
        "members": meta.members,
        "avg_daily_kwh_30d": round(avg_kwh, 2),
        "data_days": len(days),
    }
    summary = (
        f"{household_id}: {meta.house_type or '주택'} {meta.residential_area or '?'}㎡ "
        f"{meta.members or '?'}인 | 최근30일 평균 {avg_kwh:.2f}kWh/일"
    )
    return {"summary": summary, "raw": raw}


def get_consumption_summary(household_id: str, date_str: str | None = None) -> dict:
    """일간 전력 요약 (총량, 피크, 주사용 시간대)."""
    key = _resolve_key(household_id, date_str)
    if not key:
        return {"summary": "데이터 없음", "raw": {}}
    d = _day_lookup[key]
    total = float(d.profile_24.sum()) / 1000
    peak_h = int(np.argmax(d.profile_24))
    blocks = [
        ("새벽(0-6시)",   float(d.profile_24[0:6].sum())),
        ("오전(6-12시)",  float(d.profile_24[6:12].sum())),
        ("오후(12-18시)", float(d.profile_24[12:18].sum())),
        ("저녁(18-22시)", float(d.profile_24[18:22].sum())),
        ("심야(22-24시)", float(d.profile_24[22:24].sum())),
    ]
    dominant = max(blocks, key=lambda x: x[1])[0]
    raw = {
        "household_id": household_id, "date": str(key[1]),
        "total_kwh": round(total, 2), "peak_hour": peak_h,
        "dominant_period": dominant, "is_weekday": d.is_weekday,
    }
    summary = (
        f"{key[1]} 소비: {total:.2f}kWh | 피크 {peak_h}시 | "
        f"주사용 {dominant} | {'평일' if d.is_weekday else '주말'}"
    )
    return {"summary": summary, "raw": raw}


def get_consumption_hourly(household_id: str, date_str: str | None = None) -> dict:
    """24시간 시간별 전력 소비 배열 (Wh)."""
    key = _resolve_key(household_id, date_str)
    if not key:
        return {"summary": "데이터 없음", "raw": {}}
    d = _day_lookup[key]
    hourly = {f"{h:02d}시": round(float(d.profile_24[h]), 1) for h in range(24)}
    top3 = sorted(hourly.items(), key=lambda x: x[1], reverse=True)[:3]
    summary = (
        f"{key[1]} 시간별 | 상위 3시간: "
        + ", ".join(f"{h} {w:.0f}Wh" for h, w in top3)
    )
    return {"summary": summary, "raw": {"household_id": household_id, "date": str(key[1]), "hourly_wh": hourly}}


def get_consumption_breakdown(household_id: str, date_str: str | None = None) -> dict:
    """가전별 에너지 분해 결과."""
    key = _resolve_key(household_id, date_str)
    if not key:
        return {"summary": "데이터 없음", "raw": {}}
    d = _day_lookup[key]
    text = d.appliance_text or "사용 없음"
    raw = {"household_id": household_id, "date": str(key[1]), "appliance_breakdown": text}
    return {"summary": f"{key[1]} 가전 분해: {text[:120]}", "raw": raw}


def get_similar_households(household_id: str, date_str: str | None = None, top_k: int = 3) -> dict:
    """임베딩 코사인 유사도 기반 유사 가구 검색 (cell-7의 best_combined_emb 활용)."""
    key = _resolve_key(household_id, date_str)
    if not key or key not in best_combined_emb:
        return {"summary": "임베딩 없음", "raw": {}}
    retrieved = cosine_retrieve(key, best_combined_emb, top_k=top_k)
    similar = []
    for r_hid, r_day in retrieved:
        r_d = _day_lookup.get((r_hid, r_day))
        if not r_d:
            continue
        r_meta = all_data.get(r_hid, [None])[0]
        similar.append({
            "household_id": r_hid,
            "date": str(r_day),
            "daily_kwh": round(float(r_d.profile_24.sum()) / 1000, 2),
            "peak_hour": int(np.argmax(r_d.profile_24)),
            "members": r_meta.members if r_meta else None,
            "appliance_summary": r_d.appliance_text[:80],
        })
    summary = (
        f"{household_id} 유사 가구 {len(similar)}건: "
        + " / ".join(f'{s["household_id"]}({s["daily_kwh"]}kWh)' for s in similar)
    )
    return {"summary": summary, "raw": {"query": household_id, "similar": similar}}


# ── 툴 디스패처 ──────────────────────────────────────────────────────────
TOOL_FN_MAP = {
    "get_household_profile":     get_household_profile,
    "get_consumption_summary":   get_consumption_summary,
    "get_consumption_hourly":    get_consumption_hourly,
    "get_consumption_breakdown": get_consumption_breakdown,
    "get_similar_households":    get_similar_households,
}

def dispatch_tool(name: str, args: dict) -> str:
    fn = TOOL_FN_MAP.get(name)
    if not fn:
        return json.dumps({"error": f"unknown tool: {name}"}, ensure_ascii=False)
    result = fn(**{k: v for k, v in args.items() if v is not None})
    return json.dumps(result, ensure_ascii=False, default=str)


# ═══════════════════════════════════════════════════════════════════════════
# 2. OpenAI Function-Calling 스키마 정의
# ═══════════════════════════════════════════════════════════════════════════

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_household_profile",
            "description": "가구의 기본 메타데이터(주택유형·면적·인원)와 최근 30일 평균 소비량을 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "household_id": {"type": "string", "description": "익명화된 가구 ID (예: house_001)"},
                },
                "required": ["household_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_consumption_summary",
            "description": "특정 날짜의 일간 전력 소비 요약(총량, 피크 시간, 주사용 시간대)을 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "household_id": {"type": "string", "description": "익명화된 가구 ID"},
                    "date_str":     {"type": "string", "description": "조회 날짜 (YYYY-MM-DD). 생략 시 최신 데이터."},
                },
                "required": ["household_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_consumption_hourly",
            "description": "특정 날짜의 시간별(0~23시) 전력 소비 배열(Wh)을 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "household_id": {"type": "string", "description": "익명화된 가구 ID"},
                    "date_str":     {"type": "string", "description": "조회 날짜 (YYYY-MM-DD). 생략 시 최신 데이터."},
                },
                "required": ["household_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_consumption_breakdown",
            "description": "NILM 분해 결과: 어떤 가전제품이 언제 얼마나 사용됐는지 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "household_id": {"type": "string", "description": "익명화된 가구 ID"},
                    "date_str":     {"type": "string", "description": "조회 날짜 (YYYY-MM-DD). 생략 시 최신 데이터."},
                },
                "required": ["household_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_households",
            "description": "임베딩 유사도 기반으로 소비 패턴이 유사한 가구 목록과 그들의 절감 사례를 반환합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "household_id": {"type": "string", "description": "익명화된 가구 ID"},
                    "date_str":     {"type": "string", "description": "기준 날짜 (YYYY-MM-DD). 생략 시 최신 데이터."},
                    "top_k":        {"type": "integer", "description": "반환할 유사 가구 수 (기본 3)"},
                },
                "required": ["household_id"],
            },
        },
    },
]


# ═══════════════════════════════════════════════════════════════════════════
# 3. 시스템 프롬프트
# ═══════════════════════════════════════════════════════════════════════════

SYSTEM_PROMPT = textwrap.dedent("""
    당신은 에너지 효율화 전문 어시스턴트입니다.
    가구별 전력 소비 데이터를 분석해 절감 방법을 안내합니다.

    ## 원칙
    - 모든 가구는 익명화된 ID(house_XXX)로만 식별합니다. 개인정보를 추론·언급하지 마세요.
    - 수치가 필요한 질문에는 반드시 툴을 호출해 실제 데이터를 확인하세요.
    - 데이터 없이 추측으로 kWh나 시간대를 답하지 마세요.
    - 사용 가능한 툴: get_household_profile, get_consumption_summary,
      get_consumption_hourly, get_consumption_breakdown, get_similar_households

    ## 답변 형식
    - 핵심 수치를 먼저 제시하고 절감 제안을 이어서 설명하세요.
    - 유사 가구 데이터를 인용할 때는 "유사 가구 평균" 형태로 익명화하세요.
""").strip()


# ═══════════════════════════════════════════════════════════════════════════
# 4. Agent 루프
# ═══════════════════════════════════════════════════════════════════════════

def run_agent(user_query: str, household_id: str, max_rounds: int = 5) -> dict:
    """Function-calling 에이전트 루프. 툴 호출 트레이스 + 최종 답변 반환."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"[가구 ID: {household_id}] {user_query}"},
    ]
    trace = []

    for round_idx in range(1, max_rounds + 1):
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            temperature=0,
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            result_str = dispatch_tool(name, args)
            result_obj = json.loads(result_str)
            trace.append({
                "round":   round_idx,
                "tool":    name,
                "args":    args,
                "summary": result_obj.get("summary", result_str[:80]),
            })
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result_str,
            })

    final_answer = ""
    last = messages[-1]
    if hasattr(last, "content"):
        final_answer = last.content or ""
    elif isinstance(last, dict):
        final_answer = last.get("content", "")

    return {"answer": final_answer, "trace": trace, "rounds": round_idx}


# ═══════════════════════════════════════════════════════════════════════════
# 5. 테스트 케이스 + Tool Precision / Recall / F1 평가
# ═══════════════════════════════════════════════════════════════════════════

# gold_tools: 이 질문에 반드시 호출해야 하는 툴 집합 (최소 기준)
TEST_CASES = [
    {
        "query":      "이번 주 피크 시간 줄이려면 어떻게 해야 해?",
        "gold_tools": {"get_consumption_summary", "get_consumption_hourly"},
        "description": "피크 시간 절감",
    },
    {
        "query":      "어제 어떤 가전이 많이 썼어?",
        "gold_tools": {"get_consumption_breakdown"},
        "description": "가전별 소비 분해",
    },
    {
        "query":      "나랑 비슷한 집은 어떻게 절감해?",
        "gold_tools": {"get_similar_households"},
        "description": "유사 가구 절감 사례",
    },
    {
        "query":      "우리 집 기본 정보랑 평소 소비량 알려줘.",
        "gold_tools": {"get_household_profile"},
        "description": "가구 프로파일 조회",
    },
    {
        "query":      "가장 전기 많이 쓰는 시간대와 그때 주범 가전이 뭔지 알고 싶어.",
        "gold_tools": {"get_consumption_hourly", "get_consumption_breakdown"},
        "description": "피크 + 가전 복합",
    },
]

import random
random.seed(42)
eval_houses = random.sample(list(all_data.keys()), min(3, len(all_data)))

print("=" * 70)
print("Tool-use Agent 프로토타입 평가")
print(f"대상 가구: {eval_houses}  |  테스트 케이스: {len(TEST_CASES)}개")
print("=" * 70)

eval_results = []

for house_id in eval_houses:
    print(f"\n▶ 가구: {house_id}")
    for tc in TEST_CASES:
        result = run_agent(tc["query"], house_id)
        called = {t["tool"] for t in result["trace"]}
        gold   = tc["gold_tools"]

        tp        = len(called & gold)
        precision = tp / len(called) if called else 0.0
        recall    = tp / len(gold)   if gold   else 1.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)

        eval_results.append({
            "house_id":    house_id,
            "description": tc["description"],
            "called":      sorted(called),
            "gold":        sorted(gold),
            "precision":   round(precision, 3),
            "recall":      round(recall, 3),
            "f1":          round(f1, 3),
            "rounds":      result["rounds"],
        })

        status = "✓" if recall == 1.0 else "✗"
        print(f"  [{status}] {tc['description']:20s}  "
              f"P={precision:.2f} R={recall:.2f} F1={f1:.2f}  "
              f"호출={sorted(called)}")
        for t in result["trace"]:
            print(f"       R{t['round']} → {t['tool']}({list(t['args'].keys())})  "
                  f"{t['summary'][:60]}")


# ── 집계 ──────────────────────────────────────────────────────────────────
eval_df = pd.DataFrame(eval_results)
agg     = eval_df.groupby("description")[["precision", "recall", "f1"]].mean().round(3)
overall = eval_df[["precision", "recall", "f1"]].mean().round(3)

print("\n\n=== 케이스별 평균 (가구 평균) ===")
print(agg.to_string())
print(f"\n전체 평균  Precision={overall['precision']:.3f}  "
      f"Recall={overall['recall']:.3f}  F1={overall['f1']:.3f}")

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

agg[["precision", "recall", "f1"]].plot(kind="bar", ax=axes[0], width=0.6)
axes[0].set_title("케이스별 Tool Precision / Recall / F1")
axes[0].set_ylabel("Score")
axes[0].tick_params(axis="x", rotation=25)
axes[0].set_ylim(0, 1.15)
axes[0].legend(loc="lower right")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
axes[0].spines[["top", "right"]].set_visible(False)

bar_colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = axes[1].bar(
    ["Precision", "Recall", "F1"],
    [overall["precision"], overall["recall"], overall["f1"]],
    color=bar_colors, width=0.5,
)
for bar, v in zip(bars, [overall["precision"], overall["recall"], overall["f1"]]):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=12, fontweight="bold")
axes[1].set_title("전체 평균")
axes[1].set_ylim(0, 1.15)
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
axes[1].spines[["top", "right"]].set_visible(False)

plt.suptitle("Tool-use Agent 평가 — Tool Precision / Recall / F1", fontsize=13)
plt.tight_layout()
plt.savefig("agent_tool_eval.png", dpi=150)
plt.show()

print("\n★ 평가 완료. agent_tool_eval.png 저장됨.")